# B2 context extension: CPU data audit + build of genuinely new pretraining data

**Kaggle settings:** Accelerator **None (CPU)**, Internet **on**, secret **`HF`** (gated course dataset).
**Add Input:** the output of your Phase-1 **data** notebook (`aarsh59/notebook08c3659751`): `train.bin`, `val.bin`, `legal_bpe.json`.

What it does (details: docstring of `data_audit.py`):
1. downloads the Phase-1 corpus (pinned to the dataset revision the Phase-1 build used, when available);
2. **replays the exact Phase-1 selection** (verbatim code of cells [17] + [19] of the Phase-1 notebook) and checks the
   replayed `train.bin` / `val.bin` byte-for-byte against your originals. Only a match counts as a reconstruction;
3. scans every High Court / Tribunal document: used before / exact duplicate / near-duplicate of training text /
   contamination with any Phase 1–7 test, dev or validation file / candidate;
4. encodes the candidates with your exact `legal_bpe.json` into `new_high_courts.bin`, `new_tribunals.bin`, writes every Phase-1
   SC / Acts training document once into `supreme_court_unique.bin` / `acts_unique.bin` (repeats, separately weightable), and per-source
   held-out evaluation files `heldout_*.bin` (never trained on);
5. writes `manifest_ctxext.json` (counts at every step, overlap histograms, contamination sources, available tokens).

Nothing is trained. Expected runtime: about 2–4 h on Kaggle CPU (dominated by the corpus scan and encoding).

## Setup: corpus, protected eval files, inputs

In [ ]:
import os, sys, glob, json, time, subprocess
WORK = "/kaggle/working"; os.chdir(WORK); sys.path.insert(0, WORK)
from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret("HF")
from huggingface_hub import hf_hub_download, list_repo_files
REPO = "Exploration-Lab/CS-781-Capstone"
PHASE1_REVISION = "89edba6a44d6bb9593fd13b6895b5cd3b06eb82e"   # snapshot recorded in the Phase-1 run configs (run_A test_path)
try:
    files = list_repo_files(REPO, repo_type="dataset", token=HF_TOKEN, revision=PHASE1_REVISION); REV = PHASE1_REVISION
except Exception as e:
    print("pinned revision unavailable, using main:", repr(e)[:200]); REV = "main"
    files = list_repo_files(REPO, repo_type="dataset", token=HF_TOKEN)
fetch = lambda f: hf_hub_download(repo_id=REPO, filename=f, repo_type="dataset", token=HF_TOKEN, revision=REV)
CORPUS = {s: fetch(f"Phase_1_Pretraining/Train/train_{s}.json") for s in ("supreme_court", "acts", "high_courts", "tribunals")}
for s, p in CORPUS.items():
    print(f"{s:14s} {os.path.getsize(p) / 1e9:6.2f} GB  {p}")
# protected (must never enter training): Phase-1 test + every test / dev / validation file of Phases 2-7 in the repo
latest = list_repo_files(REPO, repo_type="dataset", token=HF_TOKEN)        # later phases may only exist on main
is_eval = lambda f: f.endswith(".json") and not f.startswith("Phase_1_Pretraining/Train/") and any(
    k in f.lower() for k in ("test", "dev", "val"))
PROTECTED = []
for f in sorted(set(latest) | set(files)):
    if is_eval(f):
        try:
            PROTECTED.append(hf_hub_download(repo_id=REPO, filename=f, repo_type="dataset", token=HF_TOKEN))
        except Exception as e:
            print("could not fetch", f, repr(e)[:120])
print(len(PROTECTED), "protected files:", [p.split("/snapshots/")[-1] for p in PROTECTED])
find = lambda pat: sorted(glob.glob(os.path.join("/kaggle/input", "**", pat), recursive=True))
ORIG_TRAIN, ORIG_VAL, TOK = find("train.bin"), find("val.bin"), find("legal_bpe.json")
print("original train.bin:", ORIG_TRAIN, "| val.bin:", ORIG_VAL, "| tokenizer:", TOK)
assert ORIG_TRAIN and ORIG_VAL and TOK, "attach the Phase-1 data notebook output (train.bin, val.bin, legal_bpe.json)"

In [ ]:
os.makedirs('p1', exist_ok=True)

## Code (verbatim Phase-1 cells + audit)

In [ ]:
%%writefile p1/a1_tokenizer.py
# Legal byte-level BPE (your A1 tokenizer, optimised)
# tokenizer.py
#
# Byte-level BPE with an end-of-word marker (A1 design), adapted for the CS781 legal corpus.
#
# Changes vs. the A1 version (original kept in Assignment3/a1_tokenizer.py):
#   * Tokens are integer ids backed by raw bytes, not strings. Previously the byte "<", "/",
#     "w", ">" could merge into a token identical to the "</w>" marker (or to "<pad>"), and
#     decode() would then insert spurious spaces. Ids make that collision impossible.
#   * Legal-text pre-tokenisation inside each whitespace word: letter runs, digit groups of
#     at most 3, and punctuation runs are merged separately ("s.302(1)," -> s . 302 ( 1 ) ,),
#     so "court", "court," and "court." share one vocabulary entry instead of three, and
#     years / section numbers / citations don't flood the vocabulary. Only the last piece of
#     each whitespace word carries </w>, so decode(encode(x)) == " ".join(x.split()).
#   * Incremental BPE training (pair counts + inverted index + lazy max-heap): each merge
#     only touches the words that contain the merged pair, instead of recounting every pair
#     in the corpus for every merge. 32k merges on a legal sample take minutes, not days.
#   * Encoding uses merge ranks (GPT-2 style, same result as applying merges in order) and a
#     bounded per-word cache (the old lru_cache on a method also cached `self` and never freed).
#   * Extra reserved special tokens for later phases (cls/sep/mask, span-corruption sentinels).
#   * encode_many() encodes many documents in parallel worker processes.
#
# Assignment 3 contract is unchanged: train_tokenizer(train_text, vocab_size) returns an object
# with encode(text) -> list[int], decode(ids) -> str and a vocab_size attribute.

import re
import os
import heapq
import string
import json
from collections import Counter, defaultdict


# ============================================================
# SPECIAL TOKENS
# ============================================================

N_SENTINELS = 25

SPECIALS = (
    ["<pad>", "<unk>", "<s>", "</s>", "<cls>", "<sep>", "<mask>"]
    + [f"<extra_id_{i}>" for i in range(N_SENTINELS)]
)   # 32 specials: ids 0..31

MIN_COUNT = 2

# Pieces inside one whitespace word: letter runs | digit groups (<= 3) | punctuation runs | underscores.
# The alternatives cover every non-space character, so "".join(pieces) == word always holds.
PRETOKENIZE = re.compile(r"[^\W\d_]+|\d{1,3}|[^\w\s]+|_+")

MAX_CACHE_WORDS = 2_000_000


# ============================================================
# TEXT CLEANING (A1 helpers, unchanged; not used by the BPE)
# ============================================================

def check_valid_en(word):
    is_en = False
    is_hi = False
    has_digit = False

    for ch in word:
        code = ord(ch)

        # English alphabet
        if 0x0041 <= code <= 0x005A or 0x0061 <= code <= 0x007A:
            is_en = True

        # Arabic numerals
        elif 0x0030 <= code <= 0x0039:
            has_digit = True

        # Devanagari numerals
        elif 0x0966 <= code <= 0x096F:
            has_digit = True

        # Devanagari
        elif 0x0900 <= code <= 0x097F:
            is_hi = True

    if has_digit:
        return False

    return is_en and not is_hi


def clean_text_and_tokenize_en(text):
    text = str(text)
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    tokens = text.split()
    return [word for word in tokens if check_valid_en(word)]


# ============================================================
# BYTE-LEVEL BPE
# ============================================================

def split_word(word):
    """Whitespace word -> list of (piece, is_last_piece)."""
    pieces = PRETOKENIZE.findall(word) or [word]
    last = len(pieces) - 1
    return [(p, i == last) for i, p in enumerate(pieces)]


class ByteBPE:

    def __init__(self, vocab_size=16000):

        self.vocab_size = vocab_size

        # Id layout: specials | 256 bytes | </w> | one id per learned merge
        self.n_special = len(SPECIALS)
        self.byte_offset = self.n_special
        self.eow_id = self.byte_offset + 256

        self.merges = []            # list of (left_id, right_id); merge k creates id eow_id + 1 + k
        self._reset_tables()

    # --------------------------------------------------------
    # VOCAB TABLES
    # --------------------------------------------------------

    def _reset_tables(self):
        self.token_bytes = [b""] * self.n_special + [bytes([b]) for b in range(256)] + [b""]
        self.token_eow = [False] * (self.n_special + 256) + [True]
        for left, right in self.merges:
            self.token_bytes.append(self.token_bytes[left] + self.token_bytes[right])
            self.token_eow.append(self.token_eow[right])
        self.ranks = {pair: rank for rank, pair in enumerate(self.merges)}
        self.merge_to_id = {pair: self.eow_id + 1 + rank for rank, pair in enumerate(self.merges)}
        # decode table: bytes of each token, plus a space if it ends a word; specials decode to nothing
        self._decode_table = [
            b"" if i < self.n_special else tb + (b" " if eow else b"")
            for i, (tb, eow) in enumerate(zip(self.token_bytes, self.token_eow))
        ]
        self.itos = list(SPECIALS) + [self._display(i) for i in range(self.n_special, len(self.token_bytes))]
        self.stoi = {s: i for i, s in enumerate(self.itos)}
        self.vocab = self.itos
        self._cache = {}

    def _display(self, idx):
        """Human-readable token string (for inspection only; never used for encoding)."""
        text = self.token_bytes[idx].decode("utf-8", errors="backslashreplace")
        return text + ("</w>" if self.token_eow[idx] else "")

    @property
    def special_ids(self):
        return {tok: i for i, tok in enumerate(SPECIALS)}

    @property
    def eot_id(self):
        """Document separator used in the pretraining stream."""
        return SPECIALS.index("</s>")

    def get_vocab_size(self):
        return len(self.token_bytes)

    # --------------------------------------------------------
    # PIECE -> INITIAL SYMBOLS
    # --------------------------------------------------------

    def _piece_symbols(self, piece, is_last):
        symbols = [self.byte_offset + b for b in piece.encode("utf-8")]
        if is_last:
            symbols.append(self.eow_id)
        return symbols

    # --------------------------------------------------------
    # TRAIN BPE
    # --------------------------------------------------------

    @staticmethod
    def count_words(texts):
        """Whitespace-word frequencies of an iterable of texts."""
        counts = Counter()
        for text in texts:
            counts.update(str(text).split())
        return counts

    def fit(self, tokenized_corpus=None, target_vocab_size=None, min_count=MIN_COUNT,
            word_counts=None, min_word_freq=1, verbose=True):
        """Learn merges.

        tokenized_corpus: iterable of lists of whitespace words (A1/A3 interface), or
        word_counts:      a precomputed {whitespace word: frequency} mapping (faster for big corpora).
        min_word_freq:    ignore whitespace words rarer than this while learning merges (speed knob).
        """
        target = self.vocab_size if target_vocab_size is None else target_vocab_size
        self.vocab_size = target

        if word_counts is None:
            word_counts = Counter()
            for sentence_tokens in tokenized_corpus:
                word_counts.update(sentence_tokens)

        # whitespace words -> (piece, is_last) frequencies
        piece_freq = Counter()
        for word, count in word_counts.items():
            if count < min_word_freq:
                continue
            for piece in split_word(word):
                piece_freq[piece] += count

        words = [self._piece_symbols(p, last) for p, last in piece_freq]
        freqs = list(piece_freq.values())
        del piece_freq

        if verbose:
            print("=" * 60)
            print("BUILDING BYTE-LEVEL BPE")
            print("=" * 60)
            print(f"Unique pre-token pieces: {len(words):,}")
            print(f"Initial vocabulary size: {self.eow_id + 1:,}")
            print(f"Target vocabulary size: {target:,}")

        # pair -> total count, pair -> set of word indices containing it
        pair_counts = defaultdict(int)
        where = defaultdict(set)
        for wi, (symbols, f) in enumerate(zip(words, freqs)):
            for pair in zip(symbols, symbols[1:]):
                pair_counts[pair] += f
                where[pair].add(wi)

        # max-heap on count with deterministic tie-break on ids; stale entries are skipped lazily
        heap = [(-c, pair) for pair, c in pair_counts.items()]
        heapq.heapify(heap)

        self.merges = []
        next_id = self.eow_id + 1

        while next_id < target and heap:
            neg, pair = heapq.heappop(heap)
            current = pair_counts.get(pair, 0)
            if current != -neg:
                if current > 0:
                    heapq.heappush(heap, (-current, pair))
                continue
            if current < min_count:
                if verbose:
                    print(f"\nStopping: best pair frequency = {current}")
                break

            left, right = pair
            new_id = next_id
            next_id += 1
            self.merges.append(pair)
            touched = Counter()

            for wi in list(where.pop(pair, ())):
                symbols, f = words[wi], freqs[wi]
                # remove this word's old pairs
                for p in zip(symbols, symbols[1:]):
                    pair_counts[p] -= f
                    touched[p] += 0
                    s = where.get(p)
                    if s is not None:
                        s.discard(wi)
                # merge left-to-right
                merged, i, n = [], 0, len(symbols)
                while i < n:
                    if i < n - 1 and symbols[i] == left and symbols[i + 1] == right:
                        merged.append(new_id)
                        i += 2
                    else:
                        merged.append(symbols[i])
                        i += 1
                words[wi] = merged
                # add the new pairs
                for p in zip(merged, merged[1:]):
                    pair_counts[p] += f
                    touched[p] += 0
                    where[p].add(wi)

            pair_counts.pop(pair, None)
            for p in touched:
                c = pair_counts.get(p, 0)
                if c <= 0:
                    pair_counts.pop(p, None)
                    where.pop(p, None)
                elif new_id in p:
                    heapq.heappush(heap, (-c, p))   # only pairs with the new symbol can grow

            if verbose and len(self.merges) % 1000 == 0:
                print(f"Merges: {len(self.merges):,} | Vocab: {next_id:,} | Latest pair count: {current:,}")

        self._reset_tables()

        if verbose:
            print("\n" + "=" * 60)
            print("BPE TRAINING COMPLETE")
            print("=" * 60)
            print(f"Final vocabulary size: {self.get_vocab_size():,}")
            print(f"Total merges learned: {len(self.merges):,}")

        return self

    # --------------------------------------------------------
    # ENCODE
    # --------------------------------------------------------

    def _bpe(self, symbols):
        """Repeatedly merge the lowest-rank adjacent pair (== applying merges in learned order)."""
        ranks = self.ranks
        while len(symbols) > 1:
            best, best_rank = None, None
            for pair in zip(symbols, symbols[1:]):
                r = ranks.get(pair)
                if r is not None and (best_rank is None or r < best_rank):
                    best, best_rank = pair, r
            if best is None:
                break
            new_id = self.merge_to_id[best]
            left, right = best
            merged, i, n = [], 0, len(symbols)
            while i < n:
                if i < n - 1 and symbols[i] == left and symbols[i + 1] == right:
                    merged.append(new_id)
                    i += 2
                else:
                    merged.append(symbols[i])
                    i += 1
            symbols = merged
        return symbols

    def encode_word(self, word):
        ids = self._cache.get(word)
        if ids is None:
            ids = []
            for piece, is_last in split_word(word):
                ids.extend(self._bpe(self._piece_symbols(piece, is_last)))
            ids = tuple(ids)
            if len(self._cache) >= MAX_CACHE_WORDS:
                self._cache.clear()
            self._cache[word] = ids
        return ids

    def encode_tokens(self, sentence_tokens):
        ids = []
        for word in sentence_tokens:
            ids.extend(self.encode_word(word))
        return ids

    def encode(self, text):
        """text -> list of integer ids (whitespace is normalised to single spaces)."""
        cache = self._cache
        ids = []
        for word in str(text).split():
            w = cache.get(word)
            if w is None:
                w = self.encode_word(word)
            ids.extend(w)
        return ids

    def pool(self, workers=None):
        """Process pool whose workers each hold a copy of this tokenizer (and keep their word cache).
        Use with the module-level worker functions, e.g. pool.imap(encode_array_worker, texts)."""
        import multiprocessing as mp
        workers = workers or os.cpu_count() or 1
        return mp.get_context("fork").Pool(workers, initializer=_worker_init, initargs=(self,))

    def encode_many(self, texts, workers=None, chunksize=64):
        """Encode many documents in parallel processes; yields lists of ids in input order."""
        with self.pool(workers) as pool:
            yield from pool.imap(_worker_encode, texts, chunksize=chunksize)

    # --------------------------------------------------------
    # DECODE
    # --------------------------------------------------------

    def decode(self, ids):
        table = self._decode_table
        n = len(table)
        out = b"".join(table[int(i)] if 0 <= int(i) < n else b"" for i in ids)
        return out.decode("utf-8", errors="replace").strip()

    def ends_word(self, idx):
        """True if token idx closes a whitespace word (used to count generated words)."""
        return self.token_eow[idx]

    # --------------------------------------------------------
    # SAVE / LOAD
    # --------------------------------------------------------

    def to_json(self, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "format": "legal-bytebpe-v2",
                    "specials": SPECIALS,
                    "merges": self.merges,
                    "vocab_size": self.get_vocab_size(),
                    "itos": self.itos,          # readable, informational only
                },
                f,
                ensure_ascii=False,
            )
        print(f"Saved tokenizer to: {path}")

    @classmethod
    def from_json(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        if data.get("format") != "legal-bytebpe-v2":
            raise ValueError(f"{path} was not written by this tokenizer version")
        if data["specials"] != SPECIALS:
            raise ValueError("special tokens differ from this tokenizer version")
        tokenizer = cls(vocab_size=data["vocab_size"])
        tokenizer.merges = [tuple(pair) for pair in data["merges"]]
        tokenizer._reset_tables()
        return tokenizer


# worker-process helpers for encode_many (module level so they pickle)
_WORKER_TOKENIZER = None


def _worker_init(tokenizer):
    global _WORKER_TOKENIZER
    _WORKER_TOKENIZER = tokenizer


def _worker_encode(text):
    return _WORKER_TOKENIZER.encode(text)


def encode_array_worker(text):
    """(uint16 id array, whitespace word count) for one document; for tokenizer.pool().imap."""
    import numpy as np
    return np.asarray(_WORKER_TOKENIZER.encode(text), dtype=np.uint16), text.count(" ") + 1


# ============================================================
# ASSIGNMENT 3 TRAINING FUNCTION
# ============================================================

def train_tokenizer(train_text: str, vocab_size: int):
    text = str(train_text)
    print(f"Training tokenizer on {len(text):,} characters")
    tokenizer = ByteBPE(vocab_size=vocab_size)
    tokenizer.fit(
        word_counts=ByteBPE.count_words(text.splitlines()),
        target_vocab_size=vocab_size,
        min_count=MIN_COUNT,
    )
    return tokenizer


# ============================================================
# SIMPLE SMOKE TEST
# ============================================================

if __name__ == "__main__":

    print("=" * 60)
    print("TOKENIZER SMOKE TEST")
    print("=" * 60)

    train_text = """
    the court has jurisdiction over this matter under Section 302(1) of the Code.
    the court shall consider the evidence, vide AIR 1966 SC 1882, dated 18.12.1961.
    the evidence presented before the court </w> <pad> was 12,345 pages.
    """

    tokenizer = train_tokenizer(train_text, vocab_size=400)

    for test_text in [
        "the court has jurisdiction",
        "Section 302(1), AIR 1966 SC 1882 </w><pad> naïve — ₹5,00,000/- (u/s 138)",
    ]:
        ids = tokenizer.encode(test_text)
        decoded = tokenizer.decode(ids)
        print("\nInput:  ", test_text)
        print("Ids:    ", ids)
        print("Pieces: ", [tokenizer.itos[i] for i in ids])
        print("Decoded:", decoded)
        assert decoded == " ".join(test_text.split()), "round trip failed"

    print("\nVocabulary size:", tokenizer.get_vocab_size())

    tokenizer.to_json("bpe_smoke_test.json")
    loaded = ByteBPE.from_json("bpe_smoke_test.json")
    assert loaded.encode(test_text) == tokenizer.encode(test_text)
    os.remove("bpe_smoke_test.json")

    print("\n" + "=" * 60)
    print("SMOKE TEST PASSED")
    print("=" * 60)

In [ ]:
%%writefile p1/cell17_defs.py
def normalize_ws(text):
    return " ".join(text.split())


def doc_hash(text):
    return int(hashlib.md5(text[:2000].encode("utf-8", errors="ignore")).hexdigest(), 16)


def _parse_line(line):
    line = line.strip().rstrip(",")
    if not line.startswith("{"):
        return None
    try:
        return json.loads(line)
    except json.JSONDecodeError:
        return None


def iter_texts(path):
    """Stream the "text" field of a one-record-per-line JSON array file."""
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rec = _parse_line(line)
            if rec and rec.get("text"):
                yield rec["text"]


def sample_texts_random_seek(path, n_chars, seed):
    """Random byte seeks -> resync to the next line -> one record. Spreads the sample over the whole file."""
    rng = random.Random(seed)
    size = os.path.getsize(path)
    texts, seen, total, misses = [], set(), 0, 0
    with open(path, "rb") as f:
        while total < n_chars and misses < 5000:
            f.seek(rng.randrange(0, size))
            f.readline()
            raw = f.readline()
            rec = _parse_line(raw.decode("utf-8", errors="ignore")) if raw else None
            text = rec.get("text") if rec else None
            if not text:
                misses += 1
                continue
            h = doc_hash(text)
            if h in seen:
                misses += 1
                continue
            seen.add(h)
            texts.append(text)
            total += len(text)
            misses = 0
    if total < n_chars:
        print(f"  WARNING: {os.path.basename(path)} gave only {total / 1e6:.1f}M of {n_chars / 1e6:.1f}M chars")
    return texts


def fertility(tok, texts):
    n_tok = sum(len(tok.encode(t)) for t in texts)
    n_words = sum(len(t.split()) for t in texts)
    n_chars = sum(len(normalize_ws(t)) for t in texts)
    return n_tok / n_words, n_chars / n_tok

In [ ]:
%%writefile p1/cell19.py
# build train.bin / val.bin / val_prompts.json with the legal BPE (skipped if they exist)
# Same recipe as your earlier pipeline: whitespace-normalised docs; all Supreme Court (repeated up to
# SC_MAX_EPOCHS, hash-split slice held out for validation); Acts up to ACTS_MAX_EPOCHS; the rest filled with
# the most SC-like High Court / Tribunal docs (DSIR-style unigram score). Encoding runs on all CPU cores.
# Change vs. before: HC/Tribunals are scored on a random sample of HT_SCAN_FACTOR x the needed tokens instead
# of the whole 17 GB, which keeps this stage around an hour in pure Python.
SCRATCH = "/tmp/cs781_scratch"


_encode_doc = a1_tokenizer.encode_array_worker   # runs in worker processes: (uint16 ids, word count)


def encode_docs(name, texts, score_profile=None):
    """Encode docs to a scratch .bin; returns offsets, per-doc DSIR scores and stats."""
    bin_path = os.path.join(SCRATCH, f"{name}.bin")
    lengths, scores, stats = [], [], Counter()
    t0 = time.time()
    with open(bin_path, "wb") as out, tokenizer.pool(N_WORKERS) as pool:
        for arr, n_words in pool.imap(_encode_doc, texts, chunksize=8):
            stats["docs_seen"] += 1
            if name != "supreme_court" and len(arr) / n_words > MAX_FERTILITY:
                stats["dropped_fertility"] += 1
                continue
            arr.tofile(out)
            lengths.append(len(arr))
            if score_profile is not None:
                i64 = arr.astype(np.int64)
                scores.append(float(profiles["supreme_court"][i64].mean() - score_profile[i64].mean()))
            if len(lengths) % 50_000 == 0:
                print(f"  [{name}] {len(lengths):,} docs | {sum(lengths) / 1e6:,.0f}M tokens | "
                      f"{sum(lengths) / (time.time() - t0) / 1e6:.2f}M tok/s", flush=True)
    offsets = np.concatenate([[0], np.cumsum(np.asarray(lengths, dtype=np.int64))])
    stats.update(docs_kept=len(lengths), unique_tokens=int(offsets[-1]), encode_minutes=round((time.time() - t0) / 60, 2))
    print(f"[{name}] {dict(stats)}", flush=True)
    return {"bin": bin_path, "offsets": offsets, "scores": np.asarray(scores, np.float32), "stats": dict(stats)}


def clean_stream(texts, stats, val_sink=None):
    for raw in texts:
        text = normalize_ws(raw)
        if text.count(" ") + 1 < MIN_WORDS:
            stats["dropped_short"] += 1
            continue
        if val_sink is not None and doc_hash(text) % 100 < SC_VAL_PERCENT:
            val_sink.append(text)
            continue
        stats["chars"] += len(text)
        yield text


if NEED_DATA:
    os.makedirs(SCRATCH, exist_ok=True)
    free_gb = shutil.disk_usage(SCRATCH).free / 1e9
    print(f"Free scratch disk: {free_gb:.1f} GB")
    t_all = time.time()

    # unigram profiles for the DSIR-style score
    def unigram_logprobs(texts):
        counts = np.zeros(V, dtype=np.float64)
        with tokenizer.pool(N_WORKERS) as pool:
            for arr, _ in pool.imap(_encode_doc, (normalize_ws(t) for t in texts), chunksize=8):
                counts += np.bincount(arr, minlength=V)
        return np.log((counts + 1) / (counts.sum() + V)).astype(np.float32)

    profiles = {s: unigram_logprobs(sample_texts_random_seek(p, PROFILE_CHARS, seed=SEED + 2)) for s, p in SOURCES.items()}
    print(f"profiles built ({(time.time() - t_all) / 60:.1f} min)")

    encoded, stream_stats, val_texts = {}, {}, []
    for name in ("supreme_court", "acts"):
        stream_stats[name] = Counter()
        sink = val_texts if name == "supreme_court" else None
        encoded[name] = encode_docs(name, clean_stream(iter_texts(SOURCES[name]), stream_stats[name], sink))
    sc_chars_per_token = stream_stats["supreme_court"]["chars"] / encoded["supreme_court"]["stats"]["unique_tokens"]

    rng = np.random.default_rng(SEED)

    def doc_len(src, i):
        off = encoded[src]["offsets"]
        return int(off[i + 1] - off[i])

    def repeat_docs(src, budget):
        """Whole epochs over all docs, then a random subset for the fractional epoch."""
        n_docs = len(encoded[src]["offsets"]) - 1
        unique = encoded[src]["stats"]["unique_tokens"]
        full = int(budget // unique) if unique else 0
        picks, used = [(src, i) for _ in range(full) for i in range(n_docs)], full * unique
        for i in rng.permutation(n_docs):
            if used >= budget:
                break
            picks.append((src, int(i)))
            used += doc_len(src, int(i))
        return picks, used

    sc_budget = min(SC_MAX_EPOCHS * encoded["supreme_court"]["stats"]["unique_tokens"], SC_MAX_SHARE * TOTAL_TRAIN_TOKENS)
    acts_budget = min(ACTS_MAX_EPOCHS * encoded["acts"]["stats"]["unique_tokens"], ACTS_MAX_SHARE * TOTAL_TRAIN_TOKENS)
    sc_picks, sc_used = repeat_docs("supreme_court", sc_budget)
    acts_picks, acts_used = repeat_docs("acts", acts_budget)
    rest_budget = max(0, TOTAL_TRAIN_TOKENS - sc_used - acts_used)

    # High Courts + Tribunals: random sample, scored, pooled ranking, each doc used at most once
    sizes = {s: os.path.getsize(SOURCES[s]) for s in ("high_courts", "tribunals")}
    scan_chars = HT_SCAN_FACTOR * rest_budget * sc_chars_per_token
    for s in sizes:
        stream_stats[s] = Counter()
        sample = sample_texts_random_seek(SOURCES[s], int(scan_chars * sizes[s] / sum(sizes.values())), seed=SEED + 3)
        encoded[s] = encode_docs(s, clean_stream(sample, stream_stats[s]), score_profile=profiles[s])
        del sample
    pool_rank = sorted(((float(sc), s, i) for s in sizes for i, sc in enumerate(encoded[s]["scores"])), key=lambda x: -x[0])
    ht_picks, ht_used, score_cutoff = [], 0, None
    for sc, s, i in pool_rank:
        if ht_used >= rest_budget:
            break
        ht_picks.append((s, i))
        ht_used += doc_len(s, i)
        score_cutoff = sc

    all_picks = sc_picks + acts_picks + ht_picks
    order = rng.permutation(len(all_picks))

    # train.bin: shuffled docs separated by </s>
    src_maps = {s: np.memmap(encoded[s]["bin"], dtype=np.uint16, mode="r") for s in encoded}
    train_len = sum(doc_len(s, i) + 1 for s, i in all_picks)
    train_mm = np.memmap(os.path.join(DATA_DIR, "train.bin"), dtype=np.uint16, mode="w+", shape=(train_len,))
    pos = 0
    for j in order:
        s, i = all_picks[j]
        off = encoded[s]["offsets"]
        n = int(off[i + 1] - off[i])
        train_mm[pos:pos + n] = src_maps[s][off[i]:off[i + 1]]
        train_mm[pos + n] = EOT_ID
        pos += n + 1
    train_mm.flush()
    del train_mm
    print(f"train.bin: {train_len:,} tokens")

    # val.bin + test-like val prompts (512-word prompt, 128-word reference) from held-out SC docs
    with tokenizer.pool(N_WORKERS) as pool:
        val_arrs = [a for a, _ in pool.imap(_encode_doc, val_texts, chunksize=8)]
    val_arr = np.concatenate([np.append(a, np.uint16(EOT_ID)) for a in val_arrs]).astype(np.uint16)
    val_arr.tofile(os.path.join(DATA_DIR, "val.bin"))
    val_prompts, prng = {}, np.random.default_rng(SEED + 7)
    for idx in prng.permutation(len(val_texts)):
        words = val_texts[idx].split()
        if len(words) < 640:
            continue
        st = int(prng.integers(0, len(words) - 640 + 1))
        val_prompts[str(len(val_prompts))] = {"text": " ".join(words[st:st + 512]), "reference": " ".join(words[st + 512:st + 640])}
        if len(val_prompts) >= N_VAL_PROMPTS:
            break
    json.dump(val_prompts, open(os.path.join(DATA_DIR, "val_prompts.json"), "w", encoding="utf-8"), ensure_ascii=False)

    used = {"supreme_court": sc_used, "acts": acts_used}
    for s, i in ht_picks:
        used[s] = used.get(s, 0) + doc_len(s, i)
    manifest = {
        "tokenizer": "legal_bpe.json", "vocab_size": V, "eot_id": EOT_ID, "dtype": "uint16",
        "train_tokens": int(train_len), "val_tokens": int(len(val_arr)), "val_prompts": len(val_prompts),
        "test_fertility": fert, "whitespace_normalised": True, "hc_trib_score_cutoff": score_cutoff,
        "sources": {s: {**encoded[s]["stats"], "tokens_used": int(used.get(s, 0)),
                        "share": round(used.get(s, 0) / max(sum(used.values()), 1), 4)} for s in encoded},
        "build_minutes": round((time.time() - t_all) / 60, 1),
    }
    json.dump(manifest, open(os.path.join(DATA_DIR, "manifest.json"), "w"), indent=2)
    del src_maps
    shutil.rmtree(SCRATCH, ignore_errors=True)
    print(json.dumps(manifest, indent=2))
else:
    print("Using existing data:", sorted(os.listdir(DATA_DIR)))
    print(open(os.path.join(DATA_DIR, "manifest.json")).read()[:1500])

In [ ]:
%%writefile data_audit.py
#!/usr/bin/env python3
"""CPU data audit + build of genuinely new pretraining data for the B2 context extension (plumbing; declare it).

  python data_audit.py data_audit_config.json

Stages (every stage writes into cfg["out_dir"]; nothing existing is modified):
  A. REPLAY the exact Phase-1 selection: the verbatim code of capstone_phase1_arch12.ipynb cells [17] (helpers) and
     [19] (data build) is executed with the same constants, seeds and tokenizer, with one instrumentation patch that
     records each kept document's doc_hash (md5 of the first 2,000 normalised characters, the build's own key).
     The replayed train.bin is compared token-for-token (md5 + length) with the ORIGINAL train.bin: only a perfect
     match counts as a reconstruction of which documents B and B2 were trained on.
  B. SCAN every High Court and Tribunal record. Each document is classified: dropped_short (< MIN_WORDS, as in the
     build), used_in_original (hash among the selected docs), exact duplicate (hash seen in training / validation /
     earlier in the scan), near-duplicate of training text, contamination (overlap with Phase-1 test prompts, the
     SC validation slice, or any Phase 2-7 test/dev/validation file), or candidate.
     Near-duplicates: word 13-gram shingles, deterministically subsampled (hash mod sample_mod == 0); overlap = share of
     a document's sampled shingles found in the reference index. Thresholds: cfg["near_dup"].
  C. ENCODE the surviving candidates with the exact legal_bpe.json (same worker as the build; same MAX_FERTILITY drop),
     hash-splitting heldout_percent of documents into per-source held-out evaluation files (never trained on).
  C2. SC + ACTS: every training document once into supreme_court_unique.bin / acts_unique.bin (repeats only, but
      separately weightable in the mixture; the SC validation slice is excluded).
  D. MANIFEST (manifest_ctxext.json): the reconstructed original manifest, replay verification, per-source counts
     at every step, overlap distributions, contamination sources, and the token counts available for training.
"""
import glob
import hashlib
import json
import multiprocessing as mp
import os
import random
import shutil
import sys
import time
import zlib
from collections import Counter

import numpy as np

HERE = os.path.dirname(os.path.abspath(__file__))
cfg = json.load(open(sys.argv[1]))
OUT, SCR = cfg["out_dir"], cfg["scratch_dir"]
os.makedirs(OUT, exist_ok=True); os.makedirs(SCR, exist_ok=True)
sys.path.insert(0, cfg.get("p1_code_dir", os.path.join(HERE, "p1")))
import a1_tokenizer                                       # noqa: E402  (verbatim Phase-1 tokenizer code)
from a1_tokenizer import ByteBPE                          # noqa: E402

T0 = time.time()


def log(msg, **kv):
    rec = {"t_min": round((time.time() - T0) / 60, 1), "msg": msg, **kv}
    print(json.dumps(rec), flush=True)
    with open(os.path.join(OUT, "data_audit_log.jsonl"), "a") as f:
        f.write(json.dumps(rec) + "\n")


def md5_file(path, chunk=1 << 24):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


tokenizer = ByteBPE.from_json(cfg["tokenizer"])
V, EOT_ID = tokenizer.get_vocab_size(), tokenizer.eot_id
P1 = cfg["phase1_constants"]                              # cell [4] values (copied, not chosen)

# ======================================================================= A. replay
ns = {"os": os, "json": json, "time": time, "random": random, "hashlib": hashlib, "np": np, "shutil": shutil, "glob": glob,
      "Counter": Counter, "mp": mp, "a1_tokenizer": a1_tokenizer, "tokenizer": tokenizer, "V": V, "EOT_ID": EOT_ID,
      "N_WORKERS": cfg.get("workers") or os.cpu_count() or 4, "SOURCES": cfg["corpus_files"], "NEED_DATA": True,
      "DATA_DIR": os.path.join(SCR, "replay"), "fert": None, **P1}
os.makedirs(ns["DATA_DIR"], exist_ok=True)
exec(open(os.path.join(cfg.get("p1_code_dir", os.path.join(HERE, "p1")), "cell17_defs.py")).read(), ns)
cell19 = open(os.path.join(cfg.get("p1_code_dir", os.path.join(HERE, "p1")), "cell19.py")).read()
PATCHES = [   # instrumentation only: record doc_hash of every kept doc (same order as the build's offsets)
    ("    lengths, scores, stats = [], [], Counter()\n",
     "    lengths, scores, stats = [], [], Counter()\n    kept_hashes, _hl = [], []\n\n    def _tap(it):\n"
     "        for _t in it:\n            _hl.append(doc_hash(_t))\n            yield _t\n"),
    ("        for arr, n_words in pool.imap(_encode_doc, texts, chunksize=8):\n",
     "        for _k, (arr, n_words) in enumerate(pool.imap(_encode_doc, _tap(texts), chunksize=8)):\n"),
    ("            lengths.append(len(arr))\n", "            lengths.append(len(arr))\n            kept_hashes.append(_hl[_k])\n"),
    ('    return {"bin": bin_path, "offsets": offsets, "scores": np.asarray(scores, np.float32), "stats": dict(stats)}\n',
     '    return {"bin": bin_path, "offsets": offsets, "scores": np.asarray(scores, np.float32), "stats": dict(stats), "hashes": kept_hashes}\n'),
    ('SCRATCH = "/tmp/cs781_scratch"\n', f'SCRATCH = {os.path.join(SCR, "build_scratch")!r}\n'),
]
for a, b in PATCHES:
    assert cell19.count(a) == 1, f"patch anchor not found exactly once: {a[:60]!r}"
    cell19 = cell19.replace(a, b)
log("replay: start (verbatim cell [19] + doc_hash instrumentation)")
exec(cell19, ns)
enc, picks = ns["encoded"], ns["all_picks"]
replay_bin = os.path.join(ns["DATA_DIR"], "train.bin")
rep = {"replayed_train_tokens": int(ns["train_len"]), "replayed_md5": md5_file(replay_bin)}
if cfg.get("original_train_bin") and os.path.exists(cfg["original_train_bin"]):
    rep["original_train_tokens"] = int(os.path.getsize(cfg["original_train_bin"]) // 2)
    rep["original_md5"] = md5_file(cfg["original_train_bin"])
    rep["exact_match"] = rep["original_md5"] == rep["replayed_md5"]
else:
    rep["exact_match"] = None
    rep["note"] = "original train.bin not attached: replay cannot be verified"
if cfg.get("original_val_bin") and os.path.exists(cfg["original_val_bin"]):
    rep["val_exact_match"] = md5_file(cfg["original_val_bin"]) == md5_file(os.path.join(ns["DATA_DIR"], "val.bin"))
log("replay: done", **rep)
if rep["exact_match"] is False and not cfg.get("continue_if_replay_mismatch", False):
    json.dump({"replay": rep, "reconstructed_original_manifest": ns["manifest"]}, open(os.path.join(OUT, "manifest_ctxext.json"), "w"), indent=2)
    sys.exit("REPLAY MISMATCH: the reconstruction is not the data B/B2 saw (different corpus revision?). Stopping before the scan.")

picked = {s: set() for s in enc}
times_in_train = Counter()
for s, i in picks:
    h = enc[s]["hashes"][i]
    picked[s].add(h); times_in_train[(s, h)] += 1
train_hashes = set().union(*picked.values())
val_texts = ns["val_texts"]
val_hashes = {ns["doc_hash"](t) for t in val_texts}

# ======================================================================= near-duplicate index
NDC = cfg["near_dup"]
K, MOD = NDC["shingle_words"], NDC["sample_mod"]
_WORD = {}
PW = (np.uint64(1000003) ** np.arange(K - 1, -1, -1, dtype=np.uint64)).astype(np.uint64)


def fingerprints(text):
    w = text.split()
    if len(w) < K:
        return np.zeros(0, np.uint64)
    ids = np.fromiter((_WORD.get(x) if x in _WORD else _WORD.setdefault(x, zlib.crc32(x.encode()) | 1) for x in w),
                      dtype=np.uint64, count=len(w))
    n = len(ids) - K + 1
    h = np.zeros(n, np.uint64)
    for j in range(K):
        h += ids[j:j + n] * PW[j]
    h ^= h >> np.uint64(29)
    return np.unique(h[(h % np.uint64(MOD)) == 0])


def build_index(texts_iter):
    parts = [fingerprints(t) for t in texts_iter]
    return np.unique(np.concatenate(parts)) if parts else np.zeros(0, np.uint64)


def overlap(fp, index):
    # index is sorted (np.unique): binary search per shingle. np.isin would re-sort the whole index for every document
    if not (len(fp) and len(index)):
        return 0.0
    i = np.searchsorted(index, fp)
    i[i == len(index)] = 0
    return float((index[i] == fp).mean())


log("index: training text fingerprints")
norm, clean_min = ns["normalize_ws"], P1["MIN_WORDS"]


def training_texts():
    for s in ("supreme_court", "acts"):
        for raw in ns["iter_texts"](cfg["corpus_files"][s]):
            t = norm(raw)
            if t.count(" ") + 1 >= clean_min and ns["doc_hash"](t) in picked[s]:
                yield t
    for s in ("high_courts", "tribunals"):         # the selected HC/Tribunal docs came from the seeded random-seek sample
        sizes = {x: os.path.getsize(cfg["corpus_files"][x]) for x in ("high_courts", "tribunals")}
        rest = ns["rest_budget"]
        scan_chars = P1["HT_SCAN_FACTOR"] * rest * ns["sc_chars_per_token"]
        for raw in ns["sample_texts_random_seek"](cfg["corpus_files"][s], int(scan_chars * sizes[s] / sum(sizes.values())), seed=P1["SEED"] + 3):
            t = norm(raw)
            if ns["doc_hash"](t) in picked[s]:
                yield t


train_index = build_index(training_texts())
protected_texts, protected_sources = list(val_texts), Counter({"phase1_sc_validation_slice": len(val_texts)})
for path in cfg.get("protected_files", []):
    def strings(o):
        if isinstance(o, str):
            yield o
        elif isinstance(o, list):
            if o and all(isinstance(x, str) for x in o):
                yield " ".join(o)
            else:
                for x in o:
                    yield from strings(x)
        elif isinstance(o, dict):
            for x in o.values():
                yield from strings(x)
    try:
        data = json.load(open(path, encoding="utf-8"))
    except Exception as e:
        log("protected file unreadable", path=path, error=repr(e)[:200]); continue
    got = [norm(s) for s in strings(data) if len(s.split()) >= NDC["protected_min_words"]]
    protected_texts += got; protected_sources[os.path.relpath(path, cfg.get("protected_root", "/"))] = len(got)
protected_index = build_index(protected_texts)
protected_hashes = {ns["doc_hash"](t) for t in protected_texts}
log("index: done", train_fingerprints=int(len(train_index)), protected_fingerprints=int(len(protected_index)),
    protected_texts=len(protected_texts))

# ======================================================================= B. scan HC / Tribunals
def classify(text):
    fp = fingerprints(text)
    return overlap(fp, train_index), overlap(fp, protected_index), len(fp)


cand_paths, stats, ovl_hist = {}, {}, {}
bins = np.array([0, .01, .05, .1, .2, .3, .5, .7, .9, 1.0001])
for s in ("high_courts", "tribunals"):
    st = Counter(); seen_scan = set(); hist_t = np.zeros(len(bins) - 1, int); hist_p = np.zeros(len(bins) - 1, int)
    keep_path = os.path.join(SCR, f"candidates_{s}.jsonl")
    with open(keep_path, "w", encoding="utf-8") as out:
        batch = []

        def flush(batch):
            with mp.get_context("fork").Pool(ns["N_WORKERS"]) as pool:
                res = pool.map(classify, [t for t, _ in batch], chunksize=64)
            for (t, h), (ot, op, nf) in zip(batch, res):
                hist_t[np.searchsorted(bins, ot, side="right") - 1] += 1
                hist_p[np.searchsorted(bins, op, side="right") - 1] += 1
                if op >= NDC["protected_threshold"]:
                    st["contamination_near_dup"] += 1
                elif ot >= NDC["train_threshold"]:
                    st["near_dup_of_training"] += 1
                else:
                    st["candidate"] += 1
                    out.write(json.dumps({"h": str(h), "t": t}) + "\n")
        for raw in ns["iter_texts"](cfg["corpus_files"][s]):
            st["records"] += 1
            t = norm(raw)
            if t.count(" ") + 1 < clean_min:
                st["dropped_short"] += 1; continue
            h = ns["doc_hash"](t)
            if h in picked[s]:
                st["used_in_original_train"] += 1; continue
            if h in train_hashes:
                st["exact_dup_of_training"] += 1; continue
            if h in protected_hashes:
                st["contamination_exact"] += 1; continue
            if h in seen_scan:
                st["exact_dup_within_source"] += 1; continue
            seen_scan.add(h)
            st["chars_after_exact_dedup"] += len(t)
            batch.append((t, h))
            if len(batch) >= cfg.get("scan_batch_docs", 20000):
                flush(batch); batch = []
                log(f"scan {s}: progress", records=st["records"], candidates=st["candidate"], near_dup=st["near_dup_of_training"],
                    contamination=st["contamination_near_dup"] + st["contamination_exact"], used_in_original=st["used_in_original_train"])
        if batch:
            flush(batch)
    stats[s], cand_paths[s] = st, keep_path
    ovl_hist[s] = {"bins": bins.tolist(), "overlap_with_training": hist_t.tolist(), "overlap_with_protected": hist_p.tolist()}
    log(f"scan {s}: done", **st)

# ======================================================================= C. encode
held = cfg["heldout_percent"]


def heldout(text):
    return int(hashlib.md5(("ctxext-heldout:" + text[:2000]).encode("utf-8", errors="ignore")).hexdigest(), 16) % 100 < held


files = {}
for s in ("high_courts", "tribunals"):
    def texts():
        for line in open(cand_paths[s], encoding="utf-8"):
            yield json.loads(line)["t"]
    tr_path, ho_path = os.path.join(OUT, f"new_{s}.bin"), os.path.join(OUT, f"heldout_{s}.bin")
    st = stats[s]
    from collections import deque
    flags = deque()                                          # held-out flag per text, in imap (= input) order

    def tapped():
        for t in texts():
            flags.append(heldout(t))
            yield t
    with open(tr_path, "wb") as ftr, open(ho_path, "wb") as fho, tokenizer.pool(ns["N_WORKERS"]) as pool:
        for arr, n_words in pool.imap(a1_tokenizer.encode_array_worker, tapped(), chunksize=8):
            is_held = flags.popleft()
            if len(arr) / n_words > P1["MAX_FERTILITY"]:
                st["dropped_fertility"] += 1; continue
            dst = fho if is_held else ftr
            key = "heldout" if dst is fho else "train"
            arr.tofile(dst); np.array([EOT_ID], np.uint16).tofile(dst)
            st[f"{key}_docs"] += 1; st[f"{key}_tokens"] += int(len(arr)); st[f"{key}_words"] += int(n_words)
    files[s] = {"train_bin": tr_path, "heldout_bin": ho_path}
    log(f"encode {s}: done", train_docs=st["train_docs"], train_tokens=st["train_tokens"], heldout_docs=st["heldout_docs"],
        heldout_tokens=st["heldout_tokens"])

# ======================================================================= C2. SC + Acts, each training document once
# (already seen in Phase 1: only repeats are possible. Written separately so the mixture can weight SC and Acts directly;
#  the SC validation slice is excluded because only documents selected for train.bin are written.)
for s in ("supreme_court", "acts"):
    path, st = os.path.join(OUT, f"{s}_unique.bin"), Counter()

    def texts_s(s=s):
        for raw in ns["iter_texts"](cfg["corpus_files"][s]):
            t = norm(raw)
            if t.count(" ") + 1 >= clean_min:
                h = ns["doc_hash"](t)
                if h in picked[s] and h not in st:              # each selected doc once (first occurrence)
                    st[h] = 1
                    yield t
    n_docs = n_tok = 0
    with open(path, "wb") as f, tokenizer.pool(ns["N_WORKERS"]) as pool:
        for arr, _ in pool.imap(a1_tokenizer.encode_array_worker, texts_s(), chunksize=8):
            arr.tofile(f); np.array([EOT_ID], np.uint16).tofile(f); n_docs += 1; n_tok += int(len(arr))
    files[s] = {"train_bin": path}
    stats[s] = {"unique_docs_written": n_docs, "unique_tokens_written": n_tok,
                "times_seen_in_phase1_train_bin": round(sum(c for (x, _), c in times_in_train.items() if x == s) / max(1, len(picked[s])), 2)}
    log(f"unique {s}: done", **stats[s])

# ======================================================================= D. manifest
orig = ns["manifest"]
man = {
    "replay": rep,
    "reconstructed_original_manifest": orig,
    "original_selection": {
        s: {"docs_kept_after_cleaning": int(enc[s]["stats"].get("docs_kept", 0)), "unique_tokens": int(enc[s]["stats"].get("unique_tokens", 0)),
            "docs_selected": len(picked[s]), "doc_copies_in_train_bin": int(sum(c for (x, _), c in times_in_train.items() if x == s)),
            "tokens_in_train_bin": int(orig["sources"].get(s, {}).get("tokens_used", 0)),
            "dropped_short_in_scanned_part": int(ns["stream_stats"].get(s, {}).get("dropped_short", 0)),
            "dropped_fertility_in_scanned_part": int(enc[s]["stats"].get("dropped_fertility", 0))} for s in enc},
    "sc_validation_slice": {"docs": len(val_texts), "tokens": int(orig["val_tokens"])},
    "new_data_scan": {s: dict(stats[s]) for s in ("high_courts", "tribunals")},
    "sc_acts_unique_files": {s: stats[s] for s in ("supreme_court", "acts")},
    "overlap_histograms": ovl_hist,
    "near_dup_settings": NDC,
    "protected_sources": dict(protected_sources),
    "heldout_percent": held,
    "files": files,
    "tokenizer": {"path": cfg["tokenizer"], "sha256_16": hashlib.sha256(open(cfg["tokenizer"], "rb").read()).hexdigest()[:16],
                  "vocab_size": V, "eot_id": EOT_ID, "retrained": False},
    "available_new_training_tokens": int(sum(stats[s]["train_tokens"] for s in ("high_courts", "tribunals"))),
    "repeat_only_unique_tokens": {s: stats[s]["unique_tokens_written"] for s in ("supreme_court", "acts")},
    "minutes": round((time.time() - T0) / 60, 1),
}
json.dump(man, open(os.path.join(OUT, "manifest_ctxext.json"), "w"), indent=2)
log("manifest written", available_new_training_tokens_B=round(man["available_new_training_tokens"] / 1e9, 3))
shutil.rmtree(os.path.join(SCR, "replay"), ignore_errors=True)

## Config

In [ ]:
# Phase-1 constants: copied verbatim from capstone_phase1_arch12.ipynb cell [4] (needed for an exact replay; not choices)
P1 = dict(SEED=42, TOTAL_TRAIN_TOKENS=1_000_000_000, SC_MAX_EPOCHS=4.0, SC_MAX_SHARE=0.70, ACTS_MAX_EPOCHS=2.0, ACTS_MAX_SHARE=0.08,
          SC_VAL_PERCENT=2, MIN_WORDS=100, MAX_FERTILITY=2.0, N_VAL_PROMPTS=300, PROFILE_CHARS=30_000_000, HT_SCAN_FACTOR=4)
CFG = {
    "out_dir": os.path.join(WORK, "ctxdata"), "scratch_dir": "/tmp/ctxdata_scratch", "p1_code_dir": os.path.join(WORK, "p1"),
    "tokenizer": TOK[0], "corpus_files": CORPUS, "original_train_bin": ORIG_TRAIN[0], "original_val_bin": ORIG_VAL[0],
    "phase1_constants": P1, "workers": os.cpu_count(),
    # data-cleaning settings (defaults; review them in the manifest's overlap histograms):
    "heldout_percent": 1,                       # share of new HC/Tribunal documents kept out of training for evaluation
    "near_dup": {"shingle_words": 13, "sample_mod": 16,
                 "train_threshold": 0.5,        # >= this share of a document's 13-word shingles already in training -> dropped
                 "protected_threshold": 0.2,    # >= this share found in any test/dev/val text -> dropped (contamination)
                 "protected_min_words": 50},
    "protected_files": PROTECTED, "protected_root": "/root/.cache/huggingface/hub",
    "scan_batch_docs": 20000, "continue_if_replay_mismatch": False,
}
json.dump(CFG, open("data_audit_config.json", "w"), indent=2)
print("config written; dataset revision:", REV)

## Run the audit

In [ ]:
p = subprocess.run([sys.executable, "data_audit.py", "data_audit_config.json"])
print("exit code", p.returncode)

## Manifest summary

In [ ]:
m = json.load(open(os.path.join(CFG["out_dir"], "manifest_ctxext.json")))
print("replay:", json.dumps(m["replay"], indent=1))
if "new_data_scan" in m:
    import pandas as pd
    display(pd.DataFrame(m["original_selection"]).T)
    display(pd.DataFrame(m["new_data_scan"]).T)
    print("available NEW training tokens (HC + Tribunals):", m["available_new_training_tokens"] / 1e9, "B")
    print("repeat-only unique tokens (seen in Phase 1):", {k: v / 1e9 for k, v in m["repeat_only_unique_tokens"].items()}, "B")
    print("protected sources:", m["protected_sources"])
print(sorted(os.listdir(CFG["out_dir"])))